In [1]:
### Follow the code for disorder analysis Y SHO subband space ###

In [1]:
import time
import sys
import os

# Add the parent directory to sys.path
notebook_path = os.getcwd()  # Get the current working directory where the notebook is located
parent_directory = os.path.dirname(notebook_path)  # Move to the parent directory
sys.path.append(parent_directory)

# Now you can import config
from config import *
from Minimal_Model_2D import *
import time

np.set_printoptions(threshold=sys.maxsize)


In [2]:
class Minimal_Model_Subband_Basis(twoD_Minimal_Model):

    def Ham(self, dis_mag=0, uncorr_var=1, lmbda=0, seed=None, kx=0, ky=0, phi=0):

        M = bmat([
           [self.H_same_spin(dis_mag, uncorr_var, lmbda, seed, kx, ky ), self.Del_array(phi) ],
           [self.Del_array(phi).conjugate().transpose(),-self.H_same_spin(dis_mag, uncorr_var, lmbda, seed, -kx, -ky).conjugate() ]
       ]) 

        return M
    
    def U_dag_ph(self, my=1):

        M = bmat([
            [ self.U_dag(my), None],
            [None, self.U_dag(my).conjugate()],
        ])

        return M
    
    def Ham_changed_basis(self,  dis_mag=0, uncorr_var=1, lmbda=0, seed=None, kx=0, ky=0, phi=0, my=1):

        S_dag = self.U_dag_ph(my)

        S = self.U_dag_ph(my).transpose().conjugate()

        H = self.Ham(dis_mag, uncorr_var, lmbda, seed, kx, ky,phi)

        return S_dag.dot(H).dot(S)
    
    def Ham_changed_basis2(self,  dis_mag=0, uncorr_var=1, lmbda=0, seed=None, kx=0, ky=0, phi=0, my=1):

        S = self.U_num_ph(my)

        S_dag = S.conjugate().transpose()

        H = self.Ham(dis_mag, uncorr_var, lmbda, seed, kx, ky,phi)

        return S_dag.dot(H).dot(S)
    
    def Wvfn_overlap(self, V1, V2):

        ### returns overlap between wavefunction V1 and V2 ####

        V_ele_wse_product = np.multiply(np.conjugate(V1), V2)

        return np.sum(V_ele_wse_product) ### Reimann sum for 2D integration 

In [3]:
NSM = 60            # number of sm x lattice points
M = 50              # should be atleast thrice confinement length
AX = 2.7              # lattice spacing in nm
AY = 2.9              # lattice spacing in nm
DELTA = 0.165       # in meV
#DELTA = 0.00       # in meV
MEFF_SC = 0.023     # proximitized InAs
MEFF_SM = 0.023     # InAs
NSC = 80
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
#PHI = 0.0*np.pi
#ALPHAT = 400        # meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 6           # No of eigenvalues needed to be found 
#W_MIN = 50
#MY = 30

LY = M*AY//3         # Width in nm

print(M*AY, LY)
OMEGA = 2*4*hbm/LY**2/MEFF_SC


print(OMEGA/2, 3*OMEGA/2)

print('4t=', 4*hbm/AX**2/MEFF_SC)

MU_SC =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably 
MU_SM =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably

print(MU_SM)

UNCORR_VAR = 1167  ### uncorrelated variance of disorder

#LMBDA = np.array([0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,1])
#LMBDA = np.linspace(0.075, 0.2, 51)
#LMBDA = np.linspace(0.15, 0.21, 81)
LMBDA=0
SEED = 42  ## seed for the disorder 

A = Minimal_Model_Subband_Basis(NTOT,NSM,NSC,M,AX,AY,OMEGA, delta=DELTA, meff_sc=MEFF_SC, meff_sm=MEFF_SM, mu_sm = MU_SM, mu_sc=MU_SC)
H = A.Ham(phi=PHI)

E, V= eigsh(H ,k=K_NUM ,sigma=0, return_eigenvectors=True)

sorted_index = np.argsort(E)

Esorted = E[sorted_index]
Vsorted = V[:,sorted_index]

print(Esorted)

print(H.shape)

145.0 48.0
2.878350752445659 8.635052257336977
4t= 909.7009785507266
3.454020902934791
[-0.42175283 -0.24030036 -0.1352136   0.1352136   0.24030036  0.42175283]
(22000, 22000)


In [4]:
H_changed_basis = A.Ham_changed_basis2(my=1, phi= PHI)
print(H_changed_basis.shape)
E = eigsh(H_changed_basis ,k=6 ,sigma=0, return_eigenvectors=False)
#sorted_index = np.argsort(E)

#Esorted = E[sorted_index]
#Vsorted = V[:,sorted_index]

print(np.sort(E))

#Vsorted.shape

[2.87724573]
(50, 1)
[2.87724573]
(50, 1)
(440, 440)
[-0.42175283 -0.24030036 -0.1352136   0.1352136   0.24030036  0.42175283]


In [5]:
H_changed_basis = A.Ham_changed_basis(my=1, phi= PHI)
print(H_changed_basis.shape)
E = eigsh(H_changed_basis ,k=K_NUM ,sigma=0, return_eigenvectors=False)
#sorted_index = np.argsort(E)

#Esorted = E[sorted_index]
#Vsorted = V[:,sorted_index]

print(np.sort(E))


(440, 440)
[-0.4203338  -0.24023348 -0.13525135  0.13525135  0.24023348  0.4203338 ]


In [8]:
H_changed_basis = A.Ham_changed_basis(my=4, phi= PHI)
print(H_changed_basis.shape)
E = eigsh(H_changed_basis ,k=K_NUM ,sigma=0, return_eigenvectors=False)
#sorted_index = np.argsort(E)

#Esorted = E[sorted_index]
#Vsorted = V[:,sorted_index]

print(np.sort(E))


(1760, 1760)
[-0.50958673 -0.36094167  0.36094167  0.50958673]


In [7]:
H_changed_basis = A.Ham_changed_basis(my=20)
print(H_changed_basis.shape)
E = eigsh(H_changed_basis ,k=K_NUM ,sigma=0, return_eigenvectors=False)
#sorted_index = np.argsort(E)

#Esorted = E[sorted_index]
#Vsorted = V[:,sorted_index]

print(np.sort(E))

#Vsorted.shape

(8800, 8800)
[-0.50989524 -0.36063316  0.36063316  0.50989524]
